# Generating the Analysis-Ready NHANES Dataset

This notebook loads, merges, harmonizes, and selects variables from multiple NHANES survey cycles to produce an analysis-ready dataset for type 2 diabetes prediction research.

Starting from the raw NHANES `.XPT` files downloaded from the CDC website, the notebook merges participant records across demographic, anthropometric, blood pressure, lifestyle, laboratory, dietary, and questionnaire modules. It then constructs the binary diabetes outcome variable, applies the study population criteria, and selects the final set of non-diagnostic predictor variables.

The output is a single Parquet file that serves as the input for subsequent preprocessing, train-test splitting, class imbalance handling, model training, evaluation, and feature-importance analysis.

**Note:** If the `data/raw/` folder already contains the generated Parquet file, it is not necessary to run this notebook again. It can still be read as documentation of which variables were selected, how the outcome was constructed, and how the final dataset is structured.

In [1]:
import pandas as pd

from functools import reduce

## Defining NHANES Cycles and Data Files

This step defines the NHANES survey cycles and component files used to build the analysis-ready dataset. Each cycle entry contains a base URL pointing to the CDC NHANES data directory and a dictionary of selected `.xpt` files covering the variables needed for this study.

The selected files span demographics, body measurements, blood pressure, diabetes questionnaire data, smoking, alcohol use, physical activity, sleep, laboratory measurements, dietary nutrient intake, depression, health insurance, healthcare utilisation, food security, and medical history.

NHANES uses different file naming conventions across cycles. The 2011–2012, 2013–2014, and 2015–2016 cycles use the suffixes `_G`, `_H`, and `_I` respectively. The 2017–March 2020 pre-pandemic release uses the `P_` prefix instead of a suffix.

| Cycle | File suffix / prefix | Notes |
|---|---|---|
| 2011–2012 | `_G` | Standard NHANES cycle |
| 2013–2014 | `_H` | Standard NHANES cycle |
| 2015–2016 | `_I` | Standard NHANES cycle |
| 2017–March 2020 | `P_` | Special pre-pandemic release |

| Module | Purpose |
|---|---|
| `DEMO` | Demographic variables: age, sex, race/ethnicity, education, and income |
| `BMX` | Body measurements: BMI and waist circumference |
| `BPX` / `BPXO` | Blood pressure measurements |
| `DIQ` | Diabetes questionnaire — used for outcome construction only |
| `SMQ` | Smoking variables |
| `ALQ` | Alcohol use variables |
| `PAQ` | Physical activity and sedentary behaviour |
| `SLQ` | Sleep duration and sleep problem variables |
| `GHB` | HbA1c — used for outcome construction only |
| `GLU` | Fasting glucose — used for outcome construction only |
| `TCHOL` | Total cholesterol |
| `HDL` | HDL cholesterol |
| `DR1TOT` | First-day dietary nutrient intake totals |
| `MCQ` | Medical conditions questionnaire, including family history of diabetes |
| `BPQ` | Blood pressure and cholesterol diagnosis history |
| `DPQ` | PHQ-9 depression screener items |
| `HIQ` | Health insurance coverage |
| `HUQ` | Healthcare utilisation and self-rated health |
| `FSQ` | Food security questionnaire |

In [2]:
CYCLES = {
    "2011-2012": {
        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles",
        "suffix": "_G",
        "files": {
            "DEMO": "DEMO_G.xpt",
            "BMX": "BMX_G.xpt",
            "BPX": "BPX_G.xpt",
            "DIQ": "DIQ_G.xpt",
            "SMQ": "SMQ_G.xpt",
            "ALQ": "ALQ_G.xpt",
            "PAQ": "PAQ_G.xpt",
            "SLQ": "SLQ_G.xpt",
            "GHB": "GHB_G.xpt",
            "GLU": "GLU_G.xpt",
            "TCHOL": "TCHOL_G.xpt",
            "HDL": "HDL_G.xpt",
            "DR1TOT": "DR1TOT_G.xpt",
            "MCQ": "MCQ_G.xpt",
            "BPQ": "BPQ_G.xpt",
            "DPQ": "DPQ_G.xpt",
            "HIQ": "HIQ_G.xpt",
            "HUQ": "HUQ_G.xpt",
            "FSQ": "FSQ_G.xpt",
        },
    },
    "2013-2014": {
        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles",
        "suffix": "_H",
        "files": {
            "DEMO": "DEMO_H.xpt",
            "BMX": "BMX_H.xpt",
            "BPX": "BPX_H.xpt",
            "DIQ": "DIQ_H.xpt",
            "SMQ": "SMQ_H.xpt",
            "ALQ": "ALQ_H.xpt",
            "PAQ": "PAQ_H.xpt",
            "SLQ": "SLQ_H.xpt",
            "GHB": "GHB_H.xpt",
            "GLU": "GLU_H.xpt",
            "TCHOL": "TCHOL_H.xpt",
            "HDL": "HDL_H.xpt",
            "DR1TOT": "DR1TOT_H.xpt",
            "MCQ": "MCQ_H.xpt",
            "BPQ": "BPQ_H.xpt",
            "DPQ": "DPQ_H.xpt",
            "HIQ": "HIQ_H.xpt",
            "HUQ": "HUQ_H.xpt",
            "FSQ": "FSQ_H.xpt",
        },
    },
    "2015-2016": {
        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles",
        "suffix": "_I",
        "files": {
            "DEMO": "DEMO_I.xpt",
            "BMX": "BMX_I.xpt",
            "BPX": "BPX_I.xpt",
            "DIQ": "DIQ_I.xpt",
            "SMQ": "SMQ_I.xpt",
            "ALQ": "ALQ_I.xpt",
            "PAQ": "PAQ_I.xpt",
            "SLQ": "SLQ_I.xpt",
            "GHB": "GHB_I.xpt",
            "GLU": "GLU_I.xpt",
            "TCHOL": "TCHOL_I.xpt",
            "HDL": "HDL_I.xpt",
            "DR1TOT": "DR1TOT_I.xpt",
            "MCQ": "MCQ_I.xpt",
            "BPQ": "BPQ_I.xpt",
            "DPQ": "DPQ_I.xpt",
            "HIQ": "HIQ_I.xpt",
            "HUQ": "HUQ_I.xpt",
            "FSQ": "FSQ_I.xpt",
        },
    },
    "2017-March 2020": {
        "base": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles",
        "suffix": "P_",
        "files": {
            "DEMO": "P_DEMO.xpt",
            "BMX": "P_BMX.xpt",
            "BPXO": "P_BPXO.xpt",
            "DIQ": "P_DIQ.xpt",
            "SMQ": "P_SMQ.xpt",
            "ALQ": "P_ALQ.xpt",
            "PAQ": "P_PAQ.xpt",
            "SLQ": "P_SLQ.xpt",
            "GHB": "P_GHB.xpt",
            "GLU": "P_GLU.xpt",
            "TCHOL": "P_TCHOL.xpt",
            "HDL": "P_HDL.xpt",
            "DR1TOT": "P_DR1TOT.xpt",
            "MCQ": "P_MCQ.xpt",
            "BPQ": "P_BPQ.xpt",
            "DPQ": "P_DPQ.xpt",
            "HIQ": "P_HIQ.xpt",
            "HUQ": "P_HUQ.xpt",
            "FSQ": "P_FSQ.xpt",
        },
    },
}

## Defining Required, Optional, and Harmonized Variables

This step defines three groups of variables used to build the analysis dataset:

1. **Core required variables** needed for participant identification, population filtering, and diabetes outcome construction.
2. **Non-diagnostic predictor variables** capturing behavioral, socioeconomic, clinical, dietary, and mental health risk factors for type 2 diabetes without overlapping with the outcome definition.
3. **Harmonization groups** for variables whose names differ across NHANES cycles.

HbA1c, fasting glucose, self-reported diabetes diagnosis, insulin use, and diabetes medication use are used only to construct the outcome variable and are not included as predictors.

Several variables were excluded due to missing rates above 30%:

- **Detailed smoking intensity** (`SMQ040`, `SMD641`, `SMD650`): skip-pattern responses to the binary smoking flag; 57–81% missing.
- **Detailed alcohol frequency** (`ALQ121`, `ALQ142`, `ALQ170`, `ALQ130`): 41–79% missing; the binary heavy-drinking flag (`ALQ151`) is retained at 25% missing.
- **Raw PA minutes and days** (`PAD615`, `PAD630`, etc.): skip-pattern responses to binary activity flags; 59–80% missing. These variables are still loaded to compute the MET-weighted weekly activity summary but are not included as standalone predictors.
- **Sleep disorder symptoms** (`SLQ030`, `SLQ040`, `SLQ120`): skip-pattern block; 43% missing. The doctor-reported sleep trouble variable (`SLQ050`) is retained at near-zero missingness.
- **Weekend sleep hours** (`SLD013`): 65% missing.
- **BP and cholesterol medication variables** (`BPQ050A`, `BPQ100D`): 67–73% missing; the binary diagnosis flags (`BPQ020`, `BPQ080`) are retained at near-zero missingness.
- **LDL cholesterol and triglycerides** (`LBDLDL`, `LBXTR`): 58% missing, collected from the fasting subsample only; imputing over half the dataset from a non-random subsample is not defensible.
- **SNAP receipt** (`FSQ012`): 71% missing; adult and household food security categories are retained.

## Non-Diagnostic Predictor Groups

The predictor set covers the following variable groups, selected for their relevance to type 2 diabetes risk and their availability across all four NHANES cycles.

| Group | Variables | Purpose |
|---|---|---|
| Family history | `MCQ300C` | Inherited risk for type 2 diabetes |
| Alcohol | `ALQ151` | History of heavy drinking behaviour |
| Physical activity (derived summary) | `PAD680`, `physical_activity_moderate_equivalent_min_week` | Sedentary time and MET-weighted weekly activity total |
| Physical activity (raw, computation only) | `PAQ610`, `PAD615`, `PAQ625`, `PAD630`, `PAQ640`, `PAD645`, `PAQ655`, `PAD660`, `PAQ670`, `PAD675` | Loaded to compute the MET-weighted summary; not included as standalone predictors |
| Sleep | `SLQ050` | Self-reported sleep problems discussed with a doctor |
| Depression | `DPQ010`–`DPQ090` | PHQ-9 depression screener items, summed to `phq9_score` |
| Cardiometabolic history | `BPQ020`, `BPQ080` | Ever told had hypertension; ever told had high cholesterol |
| Healthcare access | `HIQ011`, `HIQ210`, `HUQ010`, `HUQ030`, `HUQ051` | Insurance status, insurance gaps, self-rated health, usual care location, and healthcare visits |
| Food security | `FSDAD`, `FSDHH` | Adult and household food security category |

## Variables Requiring Harmonization Across Cycles

Some variables differ between NHANES cycles and require separate handling before analysis. These variables are identified below and harmonized into consistent column names during the data preparation step.

| Group | Possible NHANES variables | Harmonized variable | Meaning |
|---|---|---|---|
| Sleep duration | `SLD010H`, `SLD012` | `sleep_hours` | Weekday sleep duration in hours |
| Standard systolic BP | `BPXSY1`, `BPXSY2`, `BPXSY3` | `mean_systolic_bp` | Mean systolic BP from available readings |
| Standard diastolic BP | `BPXDI1`, `BPXDI2`, `BPXDI3` | `mean_diastolic_bp` | Mean diastolic BP from available readings |
| Oscillometric systolic BP | `BPXOSY1`, `BPXOSY2`, `BPXOSY3` | `mean_systolic_bp` | Mean systolic BP from oscillometric readings |
| Oscillometric diastolic BP | `BPXODI1`, `BPXODI2`, `BPXODI3` | `mean_diastolic_bp` | Mean diastolic BP from oscillometric readings |

In [3]:
CORE_REQUIRED_COLUMNS = [
    "SEQN",
    "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR", "RIDEXPRG",
    "BMXBMI", "BMXWAIST",
    "DIQ010", "DID040", "DIQ050", "DIQ070",
    "LBXGH",
    "LBXGLU", "LBDGLUSI",
    "SMQ020",
    "PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665",
    "LBXTC", "LBDHDD",
    "DR1TKCAL", "DR1TPROT", "DR1TCARB", "DR1TSUGR",
    "DR1TFIBE", "DR1TTFAT", "DR1TCHOL",
    "MCQ300C",
    "ALQ151",
    "PAQ610", "PAD615", "PAQ625", "PAD630", "PAQ640", "PAD645",
    "PAQ655", "PAD660", "PAQ670", "PAD675", "PAD680",
    "SLQ050",
    "DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050", "DPQ060", "DPQ070", "DPQ080", "DPQ090",
    "BPQ020", "BPQ080",
    "HIQ011", "HIQ210",
    "HUQ010", "HUQ030", "HUQ051",
    "FSDAD", "FSDHH",
]

SLEEP_ALTERNATIVES = ["SLD010H", "SLD012"]
BP_OLD_SYSTOLIC = ["BPXSY1", "BPXSY2", "BPXSY3"]
BP_OLD_DIASTOLIC = ["BPXDI1", "BPXDI2", "BPXDI3"]
BP_OSC_SYSTOLIC = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
BP_OSC_DIASTOLIC = ["BPXODI1", "BPXODI2", "BPXODI3"]
PHQ9_ITEMS = ["DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050", "DPQ060", "DPQ070", "DPQ080", "DPQ090"]

## Loading and Merging NHANES Data by Cycle

This step loads the selected NHANES data files for each survey cycle. For each cycle, the code iterates through the component files and reads the `.XPT` files directly from the CDC website using `pandas.read_sas()`. Each file is checked for the presence of the participant identifier `SEQN`; files that cannot be loaded or do not contain `SEQN` are skipped.

After all available files for a cycle are loaded, they are merged into a single dataframe using a left join on `SEQN`, producing one row per participant with columns from all selected modules. A `CYCLE` column is then added to identify the survey cycle.

The printed output shows the shape of each loaded file and the shape of the merged dataset per cycle, which can be used to verify that all expected files were loaded correctly.

In [4]:
merged_cycles = {}

for cycle in CYCLES:
    print(f"Loading data for cycle: {cycle}")
    
    base_url = CYCLES[cycle]["base"]
    files = CYCLES[cycle]["files"]
    
    dfs = []
    for key, filename in files.items():
        url = f"{base_url}/{filename}"
        
        try:
            temp = pd.read_sas(url, format="xport")

        except Exception as e:
            print(f"[ERROR] Could not load {key}: {filename}")
            print(f"URL: {url}")
            print(f"Error: {e}")
            continue
        
        if "SEQN" not in temp.columns:
            print(f"[WARNING] {key} does not contain SEQN, skipping.")
            continue
            
        print(f"{key} {filename} shape={temp.shape}")
        dfs.append(temp)
    
    if not dfs:
        raise ValueError(f"No files could be loaded for cycle {cycle}")
    
    merged_df = reduce(lambda left, right: pd.merge(left, right, on="SEQN", how="left"), dfs)
    merged_df["CYCLE"] = cycle
    
    print(f"\nMerged shape for {cycle}: {merged_df.shape}")
    merged_cycles[cycle] = merged_df

Loading data for cycle: 2011-2012
DEMO DEMO_G.xpt shape=(9756, 48)
BMX BMX_G.xpt shape=(9338, 26)
BPX BPX_G.xpt shape=(9338, 27)
DIQ DIQ_G.xpt shape=(9364, 53)
SMQ SMQ_G.xpt shape=(6790, 30)
ALQ ALQ_G.xpt shape=(5615, 10)
PAQ PAQ_G.xpt shape=(9107, 21)
SLQ SLQ_G.xpt shape=(6175, 4)
GHB GHB_G.xpt shape=(6549, 2)
GLU GLU_G.xpt shape=(3239, 8)
TCHOL TCHOL_G.xpt shape=(7821, 3)
HDL HDL_G.xpt shape=(7821, 3)
DR1TOT DR1TOT_G.xpt shape=(9338, 166)
MCQ MCQ_G.xpt shape=(9364, 92)
BPQ BPQ_G.xpt shape=(6175, 15)
DPQ DPQ_G.xpt shape=(5615, 11)
HIQ HIQ_G.xpt shape=(9756, 17)
HUQ HUQ_G.xpt shape=(9756, 10)
FSQ FSQ_G.xpt shape=(9756, 39)

Merged shape for 2011-2012: (9756, 568)
Loading data for cycle: 2013-2014
DEMO DEMO_H.xpt shape=(10175, 47)
BMX BMX_H.xpt shape=(9813, 26)
BPX BPX_H.xpt shape=(9813, 23)
DIQ DIQ_H.xpt shape=(9770, 54)
SMQ SMQ_H.xpt shape=(7168, 32)
ALQ ALQ_H.xpt shape=(5924, 10)
PAQ PAQ_H.xpt shape=(9484, 96)
SLQ SLQ_H.xpt shape=(6464, 4)
GHB GHB_H.xpt shape=(6979, 2)
GLU GLU_H.xpt 

## Checking Required Columns Across NHANES Cycles

This step verifies whether the merged dataset for each cycle contains the variables required for the analysis. Variables with consistent names across cycles are checked against `REQUIRED_COLUMNS`. Variables with cycle-dependent names — sleep duration and blood pressure — are checked separately.

For each cycle, the output reports which standard columns are present, which sleep variable is available, and whether a complete set of blood pressure variables can be found. This is used to identify harmonization issues before constructing the final dataset.

In [5]:
for cycle, df in merged_cycles.items():
    print(f"\nChecking cycle: {cycle}")
    print("-" * 80)

    missing_core_cols = [col for col in CORE_REQUIRED_COLUMNS if col not in df.columns]

    if missing_core_cols:
        print(f"[WARNING] Missing core required columns: {missing_core_cols}")
    else:
        print("[OK] All core required columns are present.")

    available_sleep = [col for col in SLEEP_ALTERNATIVES if col in df.columns]

    if available_sleep:
        print(f"[OK] Sleep column available: {available_sleep}")
    else:
        print(f"[WARNING] No sleep column found. Expected one of: {SLEEP_ALTERNATIVES}")

    has_old_bp = all(col in df.columns for col in BP_OLD_SYSTOLIC + BP_OLD_DIASTOLIC)
    has_osc_bp = all(col in df.columns for col in BP_OSC_SYSTOLIC + BP_OSC_DIASTOLIC)

    if has_old_bp:
        print("[OK] Blood pressure columns available: standard BP format")
    elif has_osc_bp:
        print("[OK] Blood pressure columns available: oscillometric BP format")
    else:
        print("[WARNING] Complete blood pressure set not found.")


Checking cycle: 2011-2012
--------------------------------------------------------------------------------
[WARNING] Missing core required columns: ['HUQ051']
[OK] Sleep column available: ['SLD010H']
[OK] Blood pressure columns available: standard BP format

Checking cycle: 2013-2014
--------------------------------------------------------------------------------
[OK] All core required columns are present.
[OK] Sleep column available: ['SLD010H']
[OK] Blood pressure columns available: standard BP format

Checking cycle: 2015-2016
--------------------------------------------------------------------------------
[OK] All core required columns are present.
[OK] Sleep column available: ['SLD012']
[OK] Blood pressure columns available: standard BP format

Checking cycle: 2017-March 2020
--------------------------------------------------------------------------------
[OK] All core required columns are present.
[OK] Sleep column available: ['SLD012']
[OK] Blood pressure columns available: osc

In [6]:
nhanes_raw = pd.concat(merged_cycles.values(), ignore_index=True)

print(nhanes_raw.shape)
print(nhanes_raw["CYCLE"].value_counts())

(45462, 769)
CYCLE
2017-March 2020    15560
2013-2014          10175
2015-2016           9971
2011-2012           9756
Name: count, dtype: int64


In [7]:
from pathlib import Path

output_path = Path("../data/raw/nhanes_diabetes_raw.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

nhanes_raw.to_parquet(output_path, index=False)

print(f"Dataset saved to: {output_path}")
print(f"Rows: {nhanes_raw.shape[0]}")
print(f"Columns: {nhanes_raw.shape[1]}")

Dataset saved to: ../data/raw/nhanes_diabetes_raw.parquet
Rows: 45462
Columns: 769
